In [1]:
# =====================================================
# 0~6. 전체 파이프라인 통합
# =====================================================

def run_pipeline():
    # -----------------------
    # 0. 라이브러리 로드
    # -----------------------
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import scipy.stats as stats
    import math
    import platform
    import ast

    pd.set_option('display.float_format', "{:.2f}".format)

    if platform.system() == 'Darwin':
        plt.rcParams['font.family'] = 'AppleGothic'
    elif platform.system() == 'Windows':
        plt.rcParams['font.family'] = 'Malgun Gothic'
    else:
        plt.rcParams['font.family'] = 'NanumGothic'

    # -----------------------
    # 1. 데이터 로드
    # -----------------------
    df_port = pd.read_csv("../dataset/portfolio.csv")
    df_prof = pd.read_csv("../dataset/profile.csv")
    df_tran = pd.read_csv("../dataset/transcript.csv")
    df_menu = pd.read_csv("../dataset/starbucks_menu_260112.csv")

    print("프로모션 제공 데이터 크기:", df_port.shape)
    print("고객정보 데이터 크기:", df_prof.shape)
    print("제공 프로모션 데이터 크기:", df_tran.shape)
    print("메뉴 정보 데이터 크기:", df_menu.shape)

    # -----------------------
    # 2. Portfolio 전처리
    # -----------------------
    df_port['ch_web'] = df_port['channels'].apply(lambda x: 1 if 'web' in x else 0)
    df_port['ch_email'] = df_port['channels'].apply(lambda x: 1 if 'email' in x else 0)
    df_port['ch_mobile'] = df_port['channels'].apply(lambda x: 1 if 'mobile' in x else 0)
    df_port['ch_social'] = df_port['channels'].apply(lambda x: 1 if 'social' in x else 0)

    df_port['channel_count'] = df_port[['ch_web','ch_email','ch_mobile','ch_social']].sum(axis=1)

    df_port['reward_ratio'] = np.where(
        df_port['difficulty'] == 0,
        df_port['reward'],
        df_port['reward'] / df_port['difficulty']
    )
    df_port['offer_strength'] = df_port['reward'] - df_port['difficulty']
    df_port = df_port.rename(columns={'id': 'offer_id'})
    df_port = df_port.drop(columns=['channels'], errors='ignore')

    # -----------------------
    # 3. Transcript 전처리
    # -----------------------
    df_tran = df_tran.copy()
    df_tran['value'] = df_tran['value'].apply(ast.literal_eval)
    df_tran['amount'] = df_tran['value'].str.get('amount')
    df_tran['reward'] = df_tran['value'].str.get('reward')

    temp_id1 = df_tran['value'].str.get('offer id')
    temp_id2 = df_tran['value'].str.get('offer_id')
    df_tran['offer_id'] = temp_id1.fillna(temp_id2)

    df_tran.drop(columns=['value', 'reward'], inplace=True)
    df_tran['day'] = df_tran['time'] // 24

    # viewed 없는 completed 이벤트 플래그
    viewed_set = set(df_tran[df_tran['event']=='offer viewed'][['person','offer_id']].apply(tuple, axis=1))
    def _viewed_flag(log):
        if log['event'] != 'offer completed': return None
        return 1 if (log['person'], log['offer_id']) in viewed_set else 0
    df_tran['viewed_before_complete'] = df_tran.apply(_viewed_flag, axis=1)

    unaware = df_tran[df_tran['viewed_before_complete']==0].shape[0]
    total_completed = df_tran[df_tran['event']=='offer completed'].shape[0]
    print(f"이벤트 미인지 완료: {unaware}건 / 전체 완료: {total_completed}건 ({unaware/total_completed*100:.1f}%)")

    # -----------------------
    # 4. Profile 전처리
    # -----------------------
    df_prof = df_prof.copy()
    df_prof['age'] = df_prof['age'].replace(118, np.nan)
    df_prof['gender'] = df_prof['gender'].fillna('Unknown')
    df_prof['income'] = df_prof['income'].fillna(0)
    df_prof['is_profile_missing'] = np.where(
        (df_prof['gender']=='Unknown') & (df_prof['income']==0) & (df_prof['age'].isna()), 1, 0
    )
    df_prof['became_member_on'] = pd.to_datetime(df_prof['became_member_on'], format='%Y%m%d')
    #reference_date = df_prof['became_member_on'].max()
    #df_prof['membership_days'] = (reference_date - df_prof['became_member_on']).dt.days

    # 파생 변수 적용
    def group_age_gender(df):
        if df['gender']=='Unknown' or pd.isna(df['age']): return '미기입'
        if df['gender']=='O': return 'Others'
        if df['gender']=='M':
            if df['age']<20: return '20세 미만 남성'
            elif df['age']<30: return '20대 남성'
            elif df['age']<40: return '30대 남성'
            elif df['age']<50: return '40대 남성'
            elif df['age']<60: return '50대 남성'
            elif df['age']<70: return '60대 남성'
            else: return '60+ 남성'
        else:
            if df['age']<20: return '20세 미만 여성'
            elif df['age']<30: return '20대 여성'
            elif df['age']<40: return '30대 여성'
            elif df['age']<50: return '40대 여성'
            elif df['age']<60: return '50대 여성'
            elif df['age']<70: return '60대 여성'
            else: return '60+ 여성'

    df_prof['age_gender_group'] = df_prof.apply(group_age_gender, axis=1)

    def income_group(x):
        if x==0: return '누락'
        elif x<50000: return '5만 미만'
        elif x<75000: return '5-7.5만'
        elif x<100000: return '7.5-10만'
        else: return '10만 이상'

    df_prof['income_group'] = df_prof['income'].apply(income_group)

    #def member_period(days):
        #if pd.isna(days) or days<0: return '미기재'
        #elif days<365: return '신규 (1년 미만)'
        #elif days<730: return '중간 (1~2년)'
        #else: return '장기 (2년+)'

    #df_prof['membership_period'] = df_prof['membership_days'].apply(member_period)
    ## 기준 날짜
    #reference_date = df_prof['became_member_on'].max()
    #df_prof['membership_days'] = (
        #reference_date - df_prof['became_member_on']
    #).dt.days
    
    # -----------------------
    # 5. 데이터 병합
    # -----------------------
    df_port['offer_id'] = df_port['offer_id'].astype(str)
    df_tran['offer_id'] = df_tran['offer_id'].astype(str)
    df_prof = df_prof.rename(columns={'id': 'customer_id'})
    df_prof['customer_id'] = df_prof['customer_id'].astype(str)
    df_tran['customer_id'] = df_tran['customer_id'].astype(str)

    df = df_tran.merge(df_port, on='offer_id', how='left')
    df = df.merge(df_prof, on='customer_id', how='left')

    print("Before merge:", df_tran.shape)
    print("After merge:", df.shape)

    # -----------------------
    # 6. 저장
    # -----------------------
    df.to_csv("preprocessed_fin.csv", index=False)
    print("✅ preprocessed_fin.csv 저장 완료")

    return df

# 실행
df = run_pipeline()
df.head()

프로모션 제공 데이터 크기: (10, 7)
고객정보 데이터 크기: (17000, 6)
제공 프로모션 데이터 크기: (306534, 5)
메뉴 정보 데이터 크기: (195, 13)
이벤트 미인지 완료: 4855건 / 전체 완료: 33579건 (14.5%)


KeyError: 'customer_id'

In [2]:
df_prof.columns

NameError: name 'df_prof' is not defined